# 01 — EDA + Preprocessing (full preprocessing.md coverage, Kaggle-ready)

Business Entity Resolution: `S1 -> S2/S3`, macro-F0.5. Run top-to-bottom locally or on Kaggle.

- **Local**: data auto-found at `../6ab10eb3b23ba_student_resource/student_resource/dataset` (or `DATA_ROOT` env).
- **Kaggle**: upload TSVs as a dataset, set `DATA_ROOT=/kaggle/input/<dataset>`.
- `FULL_RUN=true` processes every row chunked into **per-country parquet** (`preprocessing.md` §4 contract). Default (`false`) runs full EDA + 100k sample validation.
- Covers preprocessing.md §2 (EDA checklist), §3 (normalization), §3.3 (native-script), §5 (done definition).
- **Hybrid transliteration (new)**: IndicXlit (native scripts) + anyascii (Latin/French), see the "Hybrid transliteration engine" section below. `XLIT_DIR` unset = byte-identical to the original baseline.


In [36]:
import os

In [37]:
os.environ["XLIT_DIR"] = "/kaggle/input/models/sameerrathore/indivik/pytorch/default/1"

In [38]:
os.environ["DATA_ROOT"] = "/kaggle/input/datasets/sameerrathore/dataset/dataset"

In [39]:
!pip install -q "pip<24.1"

In [40]:
!pip install -q indic-nlp-library

In [41]:
!pip install -q \
    polars \
    pandas \
    numpy \
    scipy \
    scikit-learn \
    rapidfuzz \
    tqdm \
    pyarrow \
    sentence-transformers \
  

In [42]:
!pip install -q anyascii

In [43]:
print("hello")

hello


In [44]:
import os
from pathlib import Path

print("=== DATASET ===")

DATA_ROOT = Path("/kaggle/input/datasets/sameerrathore/dataset/dataset")

for p in [
    DATA_ROOT / "train/train_source1.tsv",
    DATA_ROOT / "train/train_source2.tsv",
    DATA_ROOT / "train/train_source3.tsv",
    DATA_ROOT / "train/train_ground_truth.tsv",
    DATA_ROOT / "test/test_source1.tsv",
    DATA_ROOT / "test/test_source2.tsv",
    DATA_ROOT / "test/test_source3.tsv",
]:
    print(p, "->", p.exists())

print("\n=== INDICXLIT ===")

XLIT_DIR = Path("/kaggle/input/models/sameerrathore/indivik/pytorch/default/1")

print("XLIT_DIR exists:", XLIT_DIR.exists())

for p in XLIT_DIR.rglob("*"):
    if p.is_file():
        print(p)

=== DATASET ===
/kaggle/input/datasets/sameerrathore/dataset/dataset/train/train_source1.tsv -> True
/kaggle/input/datasets/sameerrathore/dataset/dataset/train/train_source2.tsv -> True
/kaggle/input/datasets/sameerrathore/dataset/dataset/train/train_source3.tsv -> True
/kaggle/input/datasets/sameerrathore/dataset/dataset/train/train_ground_truth.tsv -> True
/kaggle/input/datasets/sameerrathore/dataset/dataset/test/test_source1.tsv -> True
/kaggle/input/datasets/sameerrathore/dataset/dataset/test/test_source2.tsv -> True
/kaggle/input/datasets/sameerrathore/dataset/dataset/test/test_source3.tsv -> True

=== INDICXLIT ===
XLIT_DIR exists: True
/kaggle/input/models/sameerrathore/indivik/pytorch/default/1/transformer/indicxlit.pt
/kaggle/input/models/sameerrathore/indivik/pytorch/default/1/corpus-bin/dict.as.txt
/kaggle/input/models/sameerrathore/indivik/pytorch/default/1/corpus-bin/dict.gu.txt
/kaggle/input/models/sameerrathore/indivik/pytorch/default/1/corpus-bin/dict.kn.txt
/kaggle/inp

In [45]:
import os, re, json, unicodedata
from pathlib import Path

import polars as pl

REPO = Path.cwd()
_env_root = os.environ.get("DATA_ROOT", "").strip()
DATA_ROOT = Path(_env_root) if _env_root else None
if DATA_ROOT is None or not DATA_ROOT.exists():
    cands = [REPO / "../6ab10eb3b23ba_student_resource/student_resource/dataset",
             REPO.parent / "6ab10eb3b23ba_student_resource/student_resource/dataset",
             Path("/kaggle/input/ml-challenge-2026")]
    DATA_ROOT = next((p.resolve() for p in cands if p.exists()), cands[0].resolve())
assert Path(DATA_ROOT).exists(), f"DATA_ROOT not found: {DATA_ROOT}"
# FULL_RUN = os.environ.get("FULL_RUN", "false").lower() == "true"
FULL_RUN = False
OUT = REPO / "processed"
OUT.mkdir(parents=True, exist_ok=True)
SEP, SAMPLE_N, CHUNK = "\t", 100000, 200000
P = {"s1tr": DATA_ROOT/"train/train_source1.tsv", "s2tr": DATA_ROOT/"train/train_source2.tsv",
     "s3tr": DATA_ROOT/"train/train_source3.tsv", "gt": DATA_ROOT/"train/train_ground_truth.tsv",
     "s1te": DATA_ROOT/"test/test_source1.tsv", "s2te": DATA_ROOT/"test/test_source2.tsv",
     "s3te": DATA_ROOT/"test/test_source3.tsv"}
print("REPO=", str(REPO))
print("DATA_ROOT=", str(DATA_ROOT))
print("FULL_RUN=", FULL_RUN)
print("missing=", [k for k, v in P.items() if not Path(v).exists()] or "none")

def safe(x, n=120):
    s = repr(x)
    return s[:n]


REPO= /kaggle/working
DATA_ROOT= /kaggle/input/datasets/sameerrathore/dataset/dataset
FULL_RUN= False
missing= none


In [46]:
# ---- EDA basics: shapes, schema, prefixes, nulls, ground truth ----
eda = {}
FILES = ["s1tr", "s2tr", "s3tr", "s1te", "s2te", "s3te"]
for k in FILES:
    n = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).select(pl.len()).collect().item()
    eda[k + "_rows"] = n
    print(k, "rows=", n)
sch = pl.scan_csv(str(P["s1tr"]), separator="\t", infer_schema_length=10000).collect_schema()
eda["columns"] = sch.names()
print("columns=", eda["columns"])
PX = {"s1tr": "S1-", "s2tr": "S2-", "s3tr": "S3-", "s1te": "S1-", "s2te": "S2-", "s3te": "S3-"}
eda["prefix_bad"] = {}
for k, px in PX.items():
    bad = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).filter(
        ~pl.col("entity_id").str.starts_with(px)).select(pl.len()).collect().item()
    eda["prefix_bad"][k] = bad
print("prefix_mismatches=", eda["prefix_bad"])
for k in FILES:
    d = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).select(
        [pl.col(c).is_null().sum().alias(c) for c in ["business_name", "business_address"]]).collect().to_dicts()[0]
    eda[k + "_nulls"] = d
    print(k, "nulls=", d)
gt = pl.read_csv(str(P["gt"]), separator="\t", infer_schema_length=10000).with_columns(
    pl.col("matched_entity_ids").fill_null("").str.strip_chars().alias("m"))
gt = gt.with_columns(
    pl.when(pl.col("m") == "").then(0).otherwise(pl.col("m").str.split(",").list.len()).alias("mc"))
mc = gt["mc"]
eda["gt_mean_mc"] = round(float(mc.mean()), 3)
eda["gt_max_mc"] = int(mc.max())
eda["gt_singleton_pct"] = round(float((mc == 0).mean() * 100), 2)
eda["gt_dist"] = {int(r["mc"]): int(r["len"]) for r in gt.group_by("mc").len().sort("mc").head(12).to_dicts()}
print("mean_mc=", eda["gt_mean_mc"], "max=", eda["gt_max_mc"], "singleton%=", eda["gt_singleton_pct"])
json.dump(eda, open(OUT / "eda_summary.json", "w"), indent=1)
print("basics done")


s1tr rows= 2206821
s2tr rows= 5034616
s3tr rows= 5285603
s1te rows= 1732544
s2te rows= 4887273
s3te rows= 5082316
columns= ['entity_id', 'business_name', 'business_address', 'country']
prefix_mismatches= {'s1tr': 0, 's2tr': 0, 's3tr': 0, 's1te': 0, 's2te': 0, 's3te': 0}
s1tr nulls= {'business_name': 0, 'business_address': 0}
s2tr nulls= {'business_name': 0, 'business_address': 168967}
s3tr nulls= {'business_name': 0, 'business_address': 175916}
s1te nulls= {'business_name': 0, 'business_address': 0}
s2te nulls= {'business_name': 0, 'business_address': 129408}
s3te nulls= {'business_name': 0, 'business_address': 136098}
mean_mc= 3.461 max= 11 singleton%= 5.58
basics done


In [47]:
# ---- EDA structural: S2/S3 share, reuse=1, distractors, cross-country, lengths, FR samples ----
# S2 vs S3 share without exploding (string counting over full GT)
agg = gt.select([pl.col("m").str.count_matches("S2-").sum().alias("s2pairs"),
                 pl.col("m").str.count_matches("S3-").sum().alias("s3pairs")]).to_dicts()[0]
eda["s2_pairs"], eda["s3_pairs"] = int(agg["s2pairs"]), int(agg["s3pairs"])
tot = eda["s2_pairs"] + eda["s3_pairs"]
eda["s2_share_pct"] = round(100 * eda["s2_pairs"] / tot, 1)
eda["s3_share_pct"] = round(100 * eda["s3_pairs"] / tot, 1)
print("pairs S2=", eda["s2_pairs"], "S3=", eda["s3_pairs"], "share=", eda["s2_share_pct"], "/", eda["s3_share_pct"])
# Explode once: reuse check + distractor math
pairs = gt.select(["source1_entity_id", "m"]).filter(pl.col("m") != "").select(
    [pl.col("source1_entity_id").alias("s1"),
     pl.col("m").str.split(",").alias("lst")]).explode("lst").rename({"lst": "mid"})
vc = pairs.group_by("mid").len()
eda["reuse_max"] = int(vc["len"].max())
eda["matched_unique"] = int(vc.height)
s2s3_total = eda["s2tr_rows"] + eda["s3tr_rows"]
eda["distractor_pct"] = round(100 * (s2s3_total - eda["matched_unique"]) / s2s3_total, 2)
print("reuse_max=", eda["reuse_max"], "matched_unique=", eda["matched_unique"],
      "distractor%=", eda["distractor_pct"])
# Cross-country check: join pair countries (S1 map 2.2M + S2/S3 maps, id+country only)
s1map = pl.scan_csv(str(P["s1tr"]), separator="\t", infer_schema_length=10000).select(["entity_id", "country"]).collect()
s2map = pl.scan_csv(str(P["s2tr"]), separator="\t", infer_schema_length=10000).select(["entity_id", "country"]).collect()
s3map = pl.scan_csv(str(P["s3tr"]), separator="\t", infer_schema_length=10000).select(["entity_id", "country"]).collect()
p2 = pairs.filter(pl.col("mid").str.starts_with("S2-")).join(s1map, left_on="s1", right_on="entity_id", how="left").rename({"country": "c1"})
p2 = p2.join(s2map, left_on="mid", right_on="entity_id", how="left").rename({"country": "c2"})
p3 = pairs.filter(pl.col("mid").str.starts_with("S3-")).join(s1map, left_on="s1", right_on="entity_id", how="left").rename({"country": "c1"})
p3 = p3.join(s3map, left_on="mid", right_on="entity_id", how="left").rename({"country": "c2"})
eda["cross_country_mismatches"] = int(p2.filter(pl.col("c1") != pl.col("c2")).height + p3.filter(pl.col("c1") != pl.col("c2")).height)
eda["cross_country_checked"] = int(p2.height + p3.height)
print("cross-country mismatches=", eda["cross_country_mismatches"], "checked=", eda["cross_country_checked"])
del p2, p3
# Lengths per source (20k samples: char + token medians)
for k in ["s1tr", "s2tr", "s3tr"]:
    s = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).select(
        ["business_name", "business_address"]).head(20000).collect().with_columns(
        [pl.col("business_name").fill_null("").str.len_chars().alias("nl"),
         pl.col("business_name").fill_null("").str.split(" ").list.len().alias("nt"),
         pl.col("business_address").fill_null("").str.len_chars().alias("al")])
    eda[k + "_len"] = {"name_char_med": float(s["nl"].median()), "name_tok_med": float(s["nt"].median()),
                       "addr_char_med": float(s["al"].median())}
    print(k, eda[k + "_len"])
# Completeness per country (30k samples: null-addr rate + PIN-presence rate)
for k in ["s1tr", "s2tr", "s3tr"]:
    s = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).select(
        ["business_address", "country"]).head(30000).collect().with_columns(
        [pl.col("business_address").is_null().alias("null_addr"),
         pl.col("business_address").fill_null("").str.contains(r"\d{5,}").alias("has_pin")])
    eda[k + "_completeness"] = {r["country"]: {"null_addr_pct": round(100 * float(r["null_addr"]), 1),
        "pin_present_pct": round(100 * float(r["has_pin"]), 1)}
        for r in s.group_by("country").agg([pl.col("null_addr").mean(), pl.col("has_pin").mean()]).to_dicts()}
    print(k, eda[k + "_completeness"])
# FR samples: 5 rows per test source
for k in ["s1te", "s2te", "s3te"]:
    print("--- FR sample", k, "---")
    for r in pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).filter(
            pl.col("country") == "France").head(5).collect().to_dicts():
        print("   ", safe(r["business_name"]), "|", safe(r["business_address"]))
# S2/S3 country splits (train + test)
for k in ["s2tr", "s3tr", "s2te", "s3te"]:
    d = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).group_by("country").len().collect().sort("country")
    eda[k + "_country"] = {r["country"]: r["len"] for r in d.to_dicts()}
    print(k, eda[k + "_country"])
# Frequent tokens + legal-suffix counts (100k raw S1 names)
LEGAL_RAW = {"pvt", "private", "ltd", "limited", "corp", "corporation", "inc", "llc", "llp", "co",
             "enterprises", "sarl", "sas", "sa", "eurl", "sci", "groupe"}
tok = pl.scan_csv(str(P["s1tr"]), separator="\t", infer_schema_length=10000).select("business_name").head(
    100000).collect().with_columns(pl.col("business_name").fill_null("").str.to_lowercase().alias("n"))
toks = tok.select(pl.col("n").str.split(" ").alias("t")).explode("t").filter(
    (pl.col("t") != "") & (~pl.col("t").is_in(sorted(LEGAL_RAW))))
eda["top_tokens"] = [[r["t"], int(r["len"])] for r in toks.group_by("t").len().sort("len", descending=True).head(20).to_dicts()]
suf = tok.select(pl.col("n").str.split(" ").list.last().alias("last")).filter(pl.col("last").is_in(sorted(LEGAL_RAW)))
eda["top_legal"] = [[r["last"], int(r["len"])] for r in suf.group_by("last").len().sort("len", descending=True).head(10).to_dicts()]
print("top_tokens=", eda["top_tokens"][:8])
print("top_legal=", eda["top_legal"])
json.dump(eda, open(OUT / "eda_summary.json", "w"), indent=1)
print("structural EDA done")


pairs S2= 3693619 S3= 3944746 share= 48.4 / 51.6
reuse_max= 1 matched_unique= 7638365 distractor%= 25.99
cross-country mismatches= 0 checked= 7638365
s1tr {'name_char_med': 24.0, 'name_tok_med': 4.0, 'addr_char_med': 40.0}
s2tr {'name_char_med': 25.0, 'name_tok_med': 4.0, 'addr_char_med': 37.0}
s3tr {'name_char_med': 25.0, 'name_tok_med': 4.0, 'addr_char_med': 42.0}
s1tr {'US': {'null_addr_pct': 0.0, 'pin_present_pct': 10.9}, 'India': {'null_addr_pct': 0.0, 'pin_present_pct': 0.4}}
s2tr {'India': {'null_addr_pct': 2.9, 'pin_present_pct': 1.2}, 'US': {'null_addr_pct': 3.8, 'pin_present_pct': 12.5}}
s3tr {'US': {'null_addr_pct': 3.7, 'pin_present_pct': 11.9}, 'India': {'null_addr_pct': 3.1, 'pin_present_pct': 1.2}}
--- FR sample s1te ---
    '<< Team Ecole' | '175 Boulevard du Président Franklin Roosevelt, Bordeaux, Nouvelle-Aquitaine'
    'ZNB Club SARL' | 'Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Rue Pierre Dignac'
    'Thermal & Fils SASU' | '20 Rue Parmentier, Dunkerque, Hauts-de-

In [48]:
# ---- True-pair similarity (20k matched pairs) + 15-pair noise catalog ----
from rapidfuzz import fuzz
pr = pairs.sample(20000, seed=42)
s1ids, mids = pr["s1"].to_list(), pr["mid"].to_list()
s1rec = {r["entity_id"]: r for r in pl.scan_csv(str(P["s1tr"]), separator="\t",
    infer_schema_length=10000).filter(pl.col("entity_id").is_in(s1ids)).collect().to_dicts()}
s2need = [m for m in mids if m.startswith("S2-")]
s3need = [m for m in mids if m.startswith("S3-")]
mx = {}
for kk, pp, need in [("s2", P["s2tr"], s2need), ("s3", P["s3tr"], s3need)]:
    if need:
        for r in pl.scan_csv(str(pp), separator="\t", infer_schema_length=10000).filter(
                pl.col("entity_id").is_in(need)).collect().to_dicts():
            mx[r["entity_id"]] = r
ts, fr, aj = [], [], []
shown = 0
print("--- noise catalog (S1 vs matched S2/S3, 15 pairs) ---")
for a, b in zip(s1ids, mids):
    r1, r2 = s1rec.get(a), mx.get(b)
    if not r1 or not r2:
        continue
    n1, n2 = (r1["business_name"] or "").lower(), (r2["business_name"] or "").lower()
    d1, d2 = (r1["business_address"] or "").lower(), (r2["business_address"] or "").lower()
    ts.append(fuzz.token_set_ratio(n1, n2))
    fr.append(fuzz.ratio(n1, n2))
    t1, t2 = set(d1.replace(",", " ").split()), set(d2.replace(",", " ").split())
    aj.append(len(t1 & t2) / max(1, len(t1 | t2)))
    if shown < 15:
        print(f"[{b}]")
        print("  S1:", safe(r1["business_name"]), "|", safe(r1["business_address"]))
        print("  MT:", safe(r2["business_name"]), "|", safe(r2["business_address"]))
        shown += 1
import statistics as _st
eda["sim_20k"] = {"token_set_mean": round(_st.mean(ts), 1),
    "token_set_gt90_pct": round(100 * sum(1 for x in ts if x >= 90) / len(ts), 1),
    "token_set_lt60_pct": round(100 * sum(1 for x in ts if x < 60) / len(ts), 1),
    "fuzz_mean": round(_st.mean(fr), 1), "addr_jaccard_mean": round(_st.mean(aj), 3), "n": len(ts)}
print("sim_20k=", eda["sim_20k"])
json.dump(eda, open(OUT / "eda_summary.json", "w"), indent=1)
print("similarity EDA done")


--- noise catalog (S1 vs matched S2/S3, 15 pairs) ---
[S2-612025569]
  S1: 'Pediatric Dentistry Reliable Health Inc' | '9 Garden Way, Johnson City, TN'
  MT: 'Inc Pediatric Déntistry Reliable Health' | '9 GARDEN WAY, JOHNSON CITY, TN'
[S3-132732095]
  S1: 'Regional Return' | '101 Odnr Road, Wilmington, OH'
  MT: 'Regional  Return' | '101 Odnr Road, Wilmington, Ohio'
[S3-632299787]
  S1: 'Horton Shanghai Inc' | '409 Foley Avenue, IA, New Hampton'
  MT: 'INC HORTON SHANGHAI' | '409-413 Foley Ave, New Hampton, Iowa'
[S2-159601454]
  S1: 'Smart Marketing Pvt Ltd' | '127/675-B W Block Keshav Nagar Juhi Kalalan, Kanpur, Uttar Pradesh'
  MT: 'स्मार्ट मार्केटिंग प्रा. लि.' | 'DOOR NO 127/675-B W BLOCK KESHAV NAGAR JUHI KALALAN, KANPUR, KANPUR NAGAR, Uttar Pradesh'
[S3-570643014]
  S1: 'First Ministries' | '603 Gilbert Street, Edna, TX'
  MT: 'First Ministries LP' | '603 Gilbert St, Edna, Texas'
[S2-936440185]
  S1: 'Kelly Innovative Semiconductor Holdings' | '90 Commerce Park, Chatham, MA'
  M

## Hybrid transliteration engine (IndicXlit native-script + anyascii Latin)

Routes native-script tokens (Devanagari/Telugu/Tamil/...) through a pure-torch port of IndicXlit; Latin/French tokens keep going through `anyascii`, byte-identical to today. `XLIT_DIR` empty/unset => `XLIT_MODEL is None` => hybrid fully disabled, nothing below changes behavior. Set `XLIT_DIR` to the unzipped `indicxlit-indic-en-v1.0` checkpoint folder (`corpus-bin/dict.*.txt` + `transformer/indicxlit.pt`) to turn it on.


In [49]:
import unicodedata as _ud

def _strip_punct_ws(w):
    # Strip punctuation/separators only -- keep combining marks (vowel signs, virama).
    # \W wrongly treats those as non-word and deletes them, shrinking native words below
    # the short-token threshold and silently routing them through anyascii instead.
    return "".join(c for c in w if not _ud.category(c).startswith(("P", "Z")))

In [51]:
# ---- Hybrid transliteration: IndicXlit (native scripts) + anyascii (Latin/French) ----
# XLIT_DIR unset/empty -> hybrid disabled, pipeline behaves exactly as anyascii-only baseline.
import math, subprocess, sys, unicodedata as _ud

XLIT_DIR = os.environ.get("XLIT_DIR", "").strip()
XLIT_LANGS = os.environ.get("XLIT_LANGS", "").strip()  # empty = all supported langs (unused hook)

try:
    import torch
    import torch.nn.functional as F
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch
    import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                       and os.environ.get("XLIT_USE_GPU", "true").lower() == "true" else "cpu")
print("XLIT_DIR=", XLIT_DIR or "(unset -> hybrid disabled)", "| torch device=", DEVICE,
      "| cuda devices=", torch.cuda.device_count() if torch.cuda.is_available() else 0)

D, H, HD = 256, 4, 64
SCALE = math.sqrt(D)          # embed_scale: no_scale_embedding=False in checkpoint
EPS = 1e-5                    # fairseq LayerNorm default
PAD, EOS, UNK = 1, 2, 3       # fairseq Dictionary specials: <s>=0,<pad>=1,</s>=2,<unk>=3


def load_dict(path):
    syms = ["<s>", "<pad>", "</s>", "<unk>"]
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line:
                syms.append(line.rsplit(" ", 1)[0])
    return syms


SORTED_LANGS = sorted("as bn brx en gom gu hi kn ks mai ml mni mr ne or pa sa sd si ta te ur".split())
BASE_SRC = 780 + 4


def lang_idx(lang):
    return BASE_SRC + SORTED_LANGS.index(lang)


def sinusoid(n, device):
    half = D // 2
    freqs = torch.exp(torch.arange(half, device=device).float() * -(math.log(10000) / (half - 1)))
    pos = torch.arange(2, 2 + n, device=device).float().unsqueeze(1)
    e = pos * freqs.unsqueeze(0)
    return torch.cat([torch.sin(e), torch.cos(e)], dim=1)


def _mha(x, kv, p, mask=None):
    nq, nk = x.shape[0], kv.shape[0]
    Q = (x @ p["q"].T + p["qb"]).view(nq, H, HD).permute(1, 0, 2) * (HD ** -0.5)
    K = (kv @ p["k"].T + p["kb"]).view(nk, H, HD).permute(1, 0, 2)
    V = (kv @ p["v"].T + p["vb"]).view(nk, H, HD).permute(1, 0, 2)
    s = Q @ K.transpose(-1, -2)
    if mask is not None:
        s = s + mask
    o = (torch.softmax(s, dim=-1) @ V).permute(1, 0, 2).reshape(nq, D)
    return o @ p["o"].T + p["ob"]


def gelu_ffn(x, w1, b1, w2, b2):
    h = x @ w1.T + b1
    return (0.5 * h * (1.0 + torch.erf(h / math.sqrt(2.0)))) @ w2.T + b2


class XlitModel:
    """Pure-torch inference for IndicXlit indic->en (no fairseq needed)."""

    def __init__(self, ckpt_path, src_dict_path, tgt_dict_path, device=DEVICE):
        raw_sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)["model"]
        self.device = device
        self.sd = {k: v.to(device) for k, v in raw_sd.items()}
        self.src_syms = load_dict(src_dict_path)
        self.tgt_syms = load_dict(tgt_dict_path)
        self.src_id = {s: i for i, s in enumerate(self.src_syms)}
        assert len(self.src_syms) == 784 and len(self.tgt_syms) == 32, (
            f"dict size mismatch: src={len(self.src_syms)} tgt={len(self.tgt_syms)} "
            "-- wrong checkpoint/dicts?")
        self.nl = max(int(k.split(".")[2]) for k in self.sd if k.startswith("encoder.layers.")) + 1

    def _L(self, pre, i, name):
        s = self.sd
        return (lambda x: F.layer_norm(x, (D,), s[f"{pre}.layers.{i}.{name}.weight"],
                                        s[f"{pre}.layers.{i}.{name}.bias"], EPS))

    def _P(self, pre, i, attn):
        s = self.sd
        return {k: s[f"{pre}.layers.{i}.{attn}{suf}"]
                for k, suf in [("q", ".q_proj.weight"), ("qb", ".q_proj.bias"),
                               ("k", ".k_proj.weight"), ("kb", ".k_proj.bias"),
                               ("v", ".v_proj.weight"), ("vb", ".v_proj.bias"),
                               ("o", ".out_proj.weight"), ("ob", ".out_proj.bias")]}

    @torch.no_grad()
    def encode(self, ids):
        s = self.sd
        x = s["encoder.embed_tokens.weight"][ids] * SCALE + sinusoid(len(ids), self.device)
        x = F.layer_norm(x, (D,), s["encoder.layernorm_embedding.weight"],
                          s["encoder.layernorm_embedding.bias"], EPS)
        for i in range(self.nl):
            x = x + _mha(self._L("encoder", i, "self_attn_layer_norm")(x),
                          self._L("encoder", i, "self_attn_layer_norm")(x),
                          self._P("encoder", i, "self_attn"))
            x = x + gelu_ffn(self._L("encoder", i, "final_layer_norm")(x),
                              s[f"encoder.layers.{i}.fc1.weight"], s[f"encoder.layers.{i}.fc1.bias"],
                              s[f"encoder.layers.{i}.fc2.weight"], s[f"encoder.layers.{i}.fc2.bias"])
        return F.layer_norm(x, (D,), s["encoder.layer_norm.weight"],
                             s["encoder.layer_norm.bias"], EPS)

    @torch.no_grad()
    def _decode_logits(self, ids, enc):
        s = self.sd
        n = len(ids)
        x = s["decoder.embed_tokens.weight"][ids] * SCALE + sinusoid(n, self.device)
        x = F.layer_norm(x, (D,), s["decoder.layernorm_embedding.weight"],
                          s["decoder.layernorm_embedding.bias"], EPS)
        causal = torch.triu(torch.full((n, n), float("-inf"), device=self.device), 1) \
            .unsqueeze(0).expand(H, n, n)
        for i in range(self.nl):
            x = x + _mha(self._L("decoder", i, "self_attn_layer_norm")(x),
                          self._L("decoder", i, "self_attn_layer_norm")(x),
                          self._P("decoder", i, "self_attn"), causal)
            x = x + _mha(self._L("decoder", i, "encoder_attn_layer_norm")(x), enc,
                          self._P("decoder", i, "encoder_attn"))
            x = x + gelu_ffn(self._L("decoder", i, "final_layer_norm")(x),
                              s[f"decoder.layers.{i}.fc1.weight"], s[f"decoder.layers.{i}.fc1.bias"],
                              s[f"decoder.layers.{i}.fc2.weight"], s[f"decoder.layers.{i}.fc2.bias"])
        x = F.layer_norm(x, (D,), s["decoder.layer_norm.weight"],
                          s["decoder.layer_norm.bias"], EPS)
        out = x @ s["decoder.output_projection.weight"].T
        if "decoder.output_projection.bias" in s:
            out = out + s["decoder.output_projection.bias"]
        return out

    def encode_word(self, word, lang):
        ids = [lang_idx(lang)] + [self.src_id.get(c, UNK) for c in word] + [lang_idx(lang)]
        return self.encode(torch.tensor(ids, device=self.device))

    @torch.no_grad()
    def greedy(self, word, lang, max_len=60):
        enc = self.encode_word(word, lang)
        out = [EOS]
        for _ in range(max_len):
            nxt = int(self._decode_logits(torch.tensor(out, device=self.device), enc)[-1].argmax())
            if nxt == EOS:
                break
            out.append(nxt)
        return "".join(self.tgt_syms[i] for i in out[1:])


_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
_CLS = "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES)
RE_NATIVE = re.compile(f"[^{_CLS}]")
RE_LATIN_WORD = re.compile(f"^[{_CLS} ]*$")


def has_native_script(s):
    return bool(RE_NATIVE.search(s or ""))


SCRIPT_LANG = [((0x0900, 0x097F), "hi"), ((0x0C00, 0x0C7F), "te"), ((0x0B80, 0x0BFF), "ta"),
               ((0x0D00, 0x0D7F), "ml"), ((0x0C80, 0x0CFF), "kn"), ((0x0980, 0x09FF), "bn"),
               ((0x0A80, 0x0AFF), "gu"), ((0x0A00, 0x0A7F), "pa"), ((0x0B00, 0x0B7F), "or"),
               ((0x0D80, 0x0DFF), "si"), ((0x0600, 0x06FF), "ur")]


def script_lang(w):
    for (a, b), l in SCRIPT_LANG:
        if any(a <= ord(c) <= b for c in w):
            return l
    return None


LEGAL_ABBR_MAP = {"\u092a\u094d\u0930\u093e": "pvt", "\u0932\u093f": "ltd"}  # प्रा->pvt, लि->ltd


def find_xlit_assets(xlit_dir):
    base = Path(xlit_dir) / "corpus-bin"
    ckpt = Path(xlit_dir) / "transformer" / "indicxlit.pt"
    tgt_dict = base / "dict.en.txt"
    src_dict = next((p for p in sorted(base.glob("dict.*.txt")) if p.name != "dict.en.txt"), None)
    return ckpt, src_dict, tgt_dict


XLIT_MODEL = None
_indic_norms, _NF = {}, None
if XLIT_DIR:
    try:
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "indic-nlp-library"])
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _NF = IndicNormalizerFactory()
    ckpt, src_dict, tgt_dict = find_xlit_assets(XLIT_DIR)
    if ckpt.exists() and src_dict and tgt_dict.exists():
        XLIT_MODEL = XlitModel(str(ckpt), str(src_dict), str(tgt_dict), device=DEVICE)
        print("XLIT_MODEL loaded from", XLIT_DIR, "| layers=", XLIT_MODEL.nl)
    else:
        print("WARNING: XLIT_DIR set but assets not found under it (expected corpus-bin/dict.*.txt "
              "+ transformer/indicxlit.pt) -- falling back to anyascii-only baseline.")


def _indic_normalize(lang, w):
    if lang not in _indic_norms:
        try:
            _indic_norms[lang] = _NF.get_normalizer(lang)
        except Exception:
            _indic_norms[lang] = _NF.get_normalizer("hi")
    return _indic_norms[lang].normalize(w)


def _strip_punct_ws(w):
    # Strip punctuation/separators only -- keep combining marks (vowel signs, virama).
    return "".join(c for c in w if not _ud.category(c).startswith(("P", "Z")))


XLIT_CACHE = {}  # {raw_word: latin} -- cost is bounded by UNIQUE tokens, not rows


def xlit_to_latin(raw_s):
    if not raw_s:
        return raw_s
    out = []
    for w in raw_s.split(" "):
        if not w or RE_LATIN_WORD.match(w):
            out.append(w)
            continue
        if w in XLIT_CACHE:
            out.append(XLIT_CACHE[w])
            continue
        plain = _strip_punct_ws(w)
        if plain.lower() in LEGAL_ABBR_MAP:
            translit = LEGAL_ABBR_MAP[plain.lower()]
        elif len(plain) <= 4 or XLIT_MODEL is None:
            translit = anyascii(w) if HAS_AA else w
        else:
            lang = script_lang(w) or "hi"
            wn = _indic_normalize(lang, w)
            translit = XLIT_MODEL.greedy(wn, lang)
        XLIT_CACHE[w] = translit
        out.append(translit)
    return " ".join(out)

XLIT_DIR= /kaggle/input/models/sameerrathore/indivik/pytorch/default/1 | torch device= cuda | cuda devices= 2
XLIT_MODEL loaded from /kaggle/input/models/sameerrathore/indivik/pytorch/default/1 | layers= 6


### Acceptance gates -- run every time before trusting hybrid output
Regex probes always run. The IndicXlit anchor checks only run if `XLIT_MODEL` loaded (i.e. `XLIT_DIR` pointed at real assets). If any assert here fails, stop and recheck the engine against the cited fairseq sources -- don't try to "fix" it by tweaking weights or thresholds.


In [52]:
try:
    from anyascii import anyascii
    HAS_AA = True
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anyascii"])
    from anyascii import anyascii
    HAS_AA = True
print("anyascii available:", HAS_AA, "| e.g.", anyascii("Pr\u00edvate \u0926\u093f\u0932\u094d\u0932\u0940"))

anyascii available: True | e.g. Private dilli


In [53]:
# ---- Hybrid engine acceptance gates ----
_probes = [("smart", True), ("caf\u00e9", True),
           ("\u0c38\u0c4d\u0c2e\u0c3e\u0c30\u0c4d\u0c1f\u0c4d", False),
           ("\u0928\u092e\u0938\u094d\u0924\u0947", False)]
for w, expect_latin in _probes:
    got = bool(RE_LATIN_WORD.match(w))
    assert got == expect_latin, f"regex probe FAILED for {w!r}: got latin={got}, expected {expect_latin}"
print("regex probes PASS (smart/café -> latin, Telugu/Devanagari -> native)")

if XLIT_MODEL is not None:
    _anchors = [("\u0928\u092e\u0938\u094d\u0924\u0947", "namaste"),
                ("\u092e\u0939\u093e\u0930\u093e\u0937\u094d\u091f\u094d\u0930", "maharashtra"),
                ("\u0936\u0930\u094d\u092e\u093e", "sharma")]
    for native, expected in _anchors:
        XLIT_CACHE.pop(native, None)
        got = xlit_to_latin(native)
        print(f"  {native} -> {got!r} (expected {expected!r})")
        assert got == expected, (
            f"XLIT anchor mismatch: {native!r} -> {got!r}, expected {expected!r}. "
            "Protocol bug -- recheck the engine, don't retrain/adjust weights.")
        assert re.match(r"^[a-z]+$", got), "XLIT output must be pure [a-z] -- non-ASCII = broken post-processing"
    mixed = xlit_to_latin("One \u092b\u093e\u0907\u0928\u0947\u0902\u0938")
    assert mixed.split(" ")[0] == "One", "Latin token inside a mixed string must pass through unchanged"
    print("XLIT engine anchors PASS -- hybrid transliteration is ON")
else:
    print("XLIT_DIR not set / assets not found -- hybrid DISABLED, pipeline == anyascii-only "
          "baseline (expected until you vendor the checkpoint on Kaggle, sec 1.1 of the plan)")


regex probes PASS (smart/café -> latin, Telugu/Devanagari -> native)
  नमस्ते -> 'namaste' (expected 'namaste')
  महाराष्ट्र -> 'maharashtra' (expected 'maharashtra')
  शर्मा -> 'sharma' (expected 'sharma')
XLIT engine anchors PASS -- hybrid transliteration is ON


In [55]:
# ---- Normalization (full preprocessing.md §3; MIT-safe, no unidecode/GPL) ----

JUNK_RE = re.compile(r"(\*+|>>|--|<<|##|\[services\]|\(id:\s*\d+\)|<null>)", re.I)
HON_RE = re.compile(r"^\s*(smt|shri|mr|mrs|miss|dr)\.?\s+", re.I)
SPACE_RE = re.compile(r"\s+")
DOMAIN_RE = re.compile(r"^([\w\-]+)\.(com|in|fr|co\.in|net|org)(\.\w+)?$", re.I)
HOUSE_RE = re.compile(r"(?:\b(?:h\.?\s*no\.?|house|plot|door)\s*\.?\s*#?\s*(\d+[a-z]?)|#\s*(\d+))", re.I)
PO_RE = re.compile(r"\b(p\.?o\.?\s*box|unit\s*\d+|hn\b|door\s*no)\b[^,]*,?", re.I)
PIN5_RE = re.compile(r"(?<!\d)(\d{5})(?!\d)")
PIN6_RE = re.compile(r"(?<!\d)(\d{6})(?!\d)")
LAND_RE = re.compile(r"\b(near|opp|opposite|behind|atm|temple|mosque|church|bank|hospital|station|mall)\b", re.I)
LEGAL = {"pvt", "private", "ltd", "limited", "corp", "corporation", "inc", "llc", "llp",
         "co", "enterprises", "sarl", "sas", "sa", "eurl", "sci", "groupe", "s.a.s"}
ABBR = {"rd": "road", "st": "street", "cir": "circle", "ave": "avenue", "av": "avenue",
        "blvd": "boulevard", "bd": "boulevard", "r": "rue"}
US_CODES = {"AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN",
            "IA", "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV",
            "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN",
            "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC"}
US_NAMES = {"alabama": "AL", "alaska": "AK", "arizona": "AZ", "california": "CA", "texas": "TX",
            "florida": "FL", "new york": "NY", "ohio": "OH", "oklahoma": "OK", "maryland": "MD",
            "iowa": "IA", "north carolina": "NC"}
IN_MAP = {"MH": "maharashtra", "RJ": "rajasthan", "TN": "tamil nadu", "KA": "karnataka",
          "DL": "delhi", "WB": "west bengal", "UP": "uttar pradesh", "GJ": "gujarat"}
IN_NAMES = {"maharashtra", "rajasthan", "tamil nadu", "karnataka", "delhi", "west bengal",
            "uttar pradesh", "gujarat", "kolkata", "howrah"}
NATIVE_STATE = {"\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bc1": "tamil nadu",
                "\u092e\u0939\u093e\u0930\u093e\u0937\u094d\u091f\u094d\u0930": "maharashtra",
                "\u0926\u093f\u0932\u094d\u0932\u0940": "delhi",
                "\u092a\u0936\u094d\u091a\u093f\u092e \u092c\u0902\u0917\u093e\u0932": "west bengal",
                "\u0915\u0930\u094d\u0928\u093e\u091f\u0915": "karnataka"}

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def soundex(s):
    s = re.sub(r"[^a-z]", "", s.lower())
    if not s:
        return ""
    mapping = {"b": "1", "f": "1", "p": "1", "v": "1", "c": "2", "g": "2", "j": "2",
               "k": "2", "q": "2", "s": "2", "x": "2", "z": "2", "d": "3", "t": "3",
               "l": "4", "m": "5", "n": "5", "r": "6"}
    out = s[0].upper()
    prev = mapping.get(s[0], "0")
    for ch in s[1:]:
        if ch in "hw":
            continue
        d = mapping.get(ch, "0")
        if d != "0" and d != prev:
            out += d
            if len(out) == 4:
                break
        prev = d
    return (out + "000")[:4]

def base_clean(s):
    if not s:
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = strip_accents(s).lower()
    s = JUNK_RE.sub(" ", s)
    return SPACE_RE.sub(" ", s).strip(" ,.-")

def normalize_name(raw):
    raw = "" if raw is None else str(raw)
    if XLIT_MODEL is not None and has_native_script(raw):
        raw = xlit_to_latin(raw)  # native words -> IndicXlit, Latin words passthrough (sec 3.1)
    s = base_clean(raw)
    hon = int(bool(HON_RE.search(s)))
    s = HON_RE.sub("", s)
    m = DOMAIN_RE.match(s.replace(" ", ""))
    stem = m.group(1) if m else ""
    toks = [t.strip(".") for t in s.split(" ") if t]
    legal = [t for t in toks if t in LEGAL]
    core_toks = [t for t in toks if t not in LEGAL]
    if not core_toks:
        core_toks = toks  # name is only a legal word (e.g. "Private"); keep it as core
    core = " ".join(core_toks)
    return {"name_norm": s, "core_name": core,
            "legal_form": " ".join(t for t in toks if t in LEGAL), "domain_stem": stem,
            "name_lat": anyascii(core) if HAS_AA else core,
            "phonetic_code": soundex(core.split(" ")[0]) if core else "",
            "has_honorific": hon}

def extract_city_state(s):
    parts = [p.strip() for p in s.split(",") if p.strip()]
    city, state = "", ""
    if parts:
        last = parts[-1]
        toks = last.split()
        cand1 = toks[-1].upper().strip(".") if toks else ""
        cand2 = " ".join(toks[-2:]) if len(toks) >= 2 else ""
        if cand1 in US_CODES:
            state = cand1
        elif cand2 in US_NAMES:
            state = US_NAMES[cand2]
        elif cand2 in IN_NAMES or cand2 in IN_MAP.values():
            state = cand2
        elif toks and toks[-1].upper() in IN_MAP:
            state = IN_MAP[toks[-1].upper()]
        else:
            for tok in toks:
                if tok in NATIVE_STATE:
                    state = NATIVE_STATE[tok]
                    break
        if state and len(parts) >= 2:
            city = re.sub(r"\d+", "", parts[-2]).strip()
    return city, state

def normalize_address(raw):
    raw = "" if raw is None else str(raw)
    if XLIT_MODEL is not None and has_native_script(raw):
        raw = xlit_to_latin(raw)  # same hook as normalize_name (sec 3.1)
    s = base_clean(raw)
    mh = HOUSE_RE.search(s)
    house = (mh.group(1) or mh.group(2) or "").lower() if mh else ""
    m6, m5 = PIN6_RE.search(s), PIN5_RE.search(s)
    pin = m6.group(1) if m6 else (m5.group(1) if m5 else "")
    land = int(bool(LAND_RE.search(s)))
    s2 = PO_RE.sub(" ", s)
    toks = [ABBR.get(t.strip("."), t.strip(".")) for t in s2.replace(",", " ").split()]
    norm = SPACE_RE.sub(" ", " ".join(toks)).strip()
    city, state = extract_city_state(s)
    return {"addr_norm": norm, "street_tokens": " ".join(sorted(set(norm.split()))),
            "house_no": house, "zip_pin_cp": pin, "city": city, "state_code": state,
            "empty_addr": int(norm == ""), "missing_pin": int(pin == ""),
            "missing_city": int(city == ""), "landmark_flag": land}

assert normalize_name("*** Pr\u00edvate *** [Services] (ID: 1072)")["core_name"] == "private"
assert normalize_name("GURU INFOTECH PVT. LTD.")["core_name"] == "guru infotech"
assert normalize_name("SARL Dupont")["legal_form"] == "sarl"
assert normalize_name("projectsmanipalindia.com")["domain_stem"] == "projectsmanipalindia"
assert normalize_name("Shri Ram Traders")["has_honorific"] == 1
assert normalize_address("H.no 780, R. Voltaire, 75011 Paris")["house_no"] == "780"
assert normalize_address("H.no 780, R. Voltaire, 75011 Paris")["zip_pin_cp"] == "75011"
assert normalize_address("175 Boulevard Bd Saint-Germain, Paris")["addr_norm"].find("boulevard") >= 0
assert normalize_address("#12, Lake Town, Kolkata")["house_no"] == "12"
assert normalize_address("Shop 5, Near SBI ATM, Pune")["landmark_flag"] == 1
assert normalize_address("")["empty_addr"] == 1
if HAS_AA:
    assert anyascii("\u0926\u093f\u0932\u094d\u0932\u0940").strip() != ""
print("all normalize unit tests PASS (anyascii=%s)" % HAS_AA)

if XLIT_MODEL is not None:
    _sample_native = ("\u0936\u094d\u0930\u0940 \u0915\u0943\u0937\u094d\u0923\u093e "
                      "\u091f\u094d\u0930\u0947\u0921\u0930\u094d\u0938")  # Shri Krishna Traders
    print("hybrid sample:", _sample_native, "->", normalize_name(_sample_native)["name_lat"])
print("normalize_name/address hybrid hook wired (sec 3.1); baseline path untouched when XLIT_DIR unset")


all normalize unit tests PASS (anyascii=True)
hybrid sample: श्री कृष्णा ट्रेडर्स -> sri krishna traders
normalize_name/address hybrid hook wired (sec 3.1); baseline path untouched when XLIT_DIR unset


### Frame-level normalization helper
`NEW_COLS` / `normalize_frame` moved up from the original "Run" cell so the anchored-sample recall checks below can call it too -- same function, same output, just defined earlier.


In [56]:
NEW_COLS = ["name_norm", "core_name", "legal_form", "domain_stem", "name_lat",
            "phonetic_code", "has_honorific", "addr_norm", "street_tokens", "house_no",
            "zip_pin_cp", "city", "state_code", "empty_addr", "missing_pin",
            "missing_city", "landmark_flag"]

def normalize_frame(df):
    names = [normalize_name(x) for x in df["business_name"].to_list()]
    addrs = [normalize_address(x) for x in df["business_address"].to_list()]
    cols = [[n["name_norm"] for n in names], [n["core_name"] for n in names],
            [n["legal_form"] for n in names], [n["domain_stem"] for n in names],
            [n["name_lat"] for n in names], [n["phonetic_code"] for n in names],
            [n["has_honorific"] for n in names], [a["addr_norm"] for a in addrs],
            [a["street_tokens"] for a in addrs], [a["house_no"] for a in addrs],
            [a["zip_pin_cp"] for a in addrs], [a["city"] for a in addrs],
            [a["state_code"] for a in addrs], [a["empty_addr"] for a in addrs],
            [a["missing_pin"] for a in addrs], [a["missing_city"] for a in addrs],
            [a["landmark_flag"] for a in addrs]]
    return df.with_columns([pl.Series(c, v) for c, v in zip(NEW_COLS, cols)])
print("normalize_frame ready:", NEW_COLS)


normalize_frame ready: ['name_norm', 'core_name', 'legal_form', 'domain_stem', 'name_lat', 'phonetic_code', 'has_honorific', 'addr_norm', 'street_tokens', 'house_no', 'zip_pin_cp', 'city', 'state_code', 'empty_addr', 'missing_pin', 'missing_city', 'landmark_flag']


## Anchored dev sampling (fixes the recall-mismatch problem)

The independent per-file sampler below (`process_sample_per_country`, further down) takes `head(quota_each)` from S1 and from S2/S3 **separately** -- there's no guarantee a GT match for an S1 row in the sample actually landed inside the S2/S3 sample too. Computing recall against the full `train_ground_truth.tsv` on top of that sampler will look artificially bad for reasons that have nothing to do with blocking quality.

`build_anchored_sample` fixes this at the source: sample S1 first, then **pull in every true GT match** for those S1 ids from S2/S3 (guaranteed present), padded with random same-pool distractors so blocking still has real negative examples to reject. Recall computed on this sample is directly meaningful -- no post-hoc GT filtering needed.


In [58]:
# ---- Anchored dev sample: guarantees every true GT match is present in the sample ----
def build_anchored_sample(n_s1=100_000, distractor_ratio=3, seed=42, tag=None):
    s1_full = pl.scan_csv(str(P["s1tr"]), separator="\t", infer_schema_length=10000) \
        .select(["entity_id", "country"]).collect()
    n_by_country = s1_full.group_by("country").len()
    parts = []
    for r in n_by_country.to_dicts():
        c, pool_n = r["country"], r["len"]
        take = min(pool_n, max(1, round(n_s1 * pool_n / s1_full.height)))
        parts.append(s1_full.filter(pl.col("country") == c).sample(take, seed=seed, with_replacement=False))
    s1_ids = pl.concat(parts, how="diagonal").unique(subset=["entity_id"]).head(n_s1)["entity_id"].to_list()

    gt_full = pl.read_csv(str(P["gt"]), separator="\t", infer_schema_length=10000)
    gt_sample = gt_full.filter(pl.col("source1_entity_id").is_in(s1_ids))
    m = gt_sample.filter(pl.col("matched_entity_ids").is_not_null() & (pl.col("matched_entity_ids") != "")) \
        .select(pl.col("matched_entity_ids").str.split(",").alias("m")).explode("m")["m"].to_list()
    true_s2 = sorted({x for x in m if x.startswith("S2-")})
    true_s3 = sorted({x for x in m if x.startswith("S3-")})

    def pool(src, true_ids, n_distractor, seed):
        lf = pl.scan_csv(str(src), separator="\t", infer_schema_length=10000)
        true_ids_set = set(true_ids)
        true_df = lf.filter(pl.col("entity_id").is_in(true_ids)).collect() if true_ids \
            else pl.DataFrame(schema=lf.collect_schema())
        distractor_df = true_df.head(0)
        if n_distractor > 0:
            total_n = lf.select(pl.len()).collect().item()
            # hash-based LAZY sub-sample so we don't materialize the whole source file just to
            # draw a few thousand distractors -- matters once s2/s3 run into the millions of rows
            frac = min(1.0, (n_distractor / max(1, total_n)) * 2.0)
            bucket = int(frac * 1_000_000)
            distractor_df = lf.filter(~pl.col("entity_id").is_in(true_ids)) \
                .filter((pl.col("entity_id").hash(seed=seed) % 1_000_000) < bucket).collect()
            if distractor_df.height > n_distractor:
                distractor_df = distractor_df.sample(n_distractor, seed=seed)
        combined = pl.concat([true_df, distractor_df], how="diagonal").unique(subset=["entity_id"])
        assert true_ids_set <= set(combined["entity_id"].to_list()), \
            f"data integrity: GT references ids missing from {src}"
        return combined

    n_dist = len(s1_ids) * distractor_ratio
    s1_sample = pl.scan_csv(str(P["s1tr"]), separator="\t", infer_schema_length=10000) \
        .filter(pl.col("entity_id").is_in(s1_ids)).collect()
    s2_sample = pool(P["s2tr"], true_s2, n_dist, seed)
    s3_sample = pool(P["s3tr"], true_s3, n_dist, seed)

    if tag:
        d = OUT / "dev_samples" / tag
        d.mkdir(parents=True, exist_ok=True)
        s1_sample.write_parquet(str(d / "s1.parquet"))
        s2_sample.write_parquet(str(d / "s2.parquet"))
        s3_sample.write_parquet(str(d / "s3.parquet"))
        gt_sample.write_parquet(str(d / "gt.parquet"))
    print(f"anchored sample n_s1={len(s1_ids)}: s2={s2_sample.height} (true={len(true_s2)}), "
          f"s3={s3_sample.height} (true={len(true_s3)}) -- every true match is guaranteed present")
    return s1_sample, s2_sample, s3_sample, gt_sample


## Baseline blocking + recall, hybrid ON vs OFF, native-script vs Latin split

This blocking key (`country | phonetic_code | core_name[:4]`) is a deliberately simple stand-in for the real embedding-bucket step (notebook 02) -- fast enough to run here, and good enough to tell you whether the hybrid transliteration is actually moving recall on native-script S1 records before you invest in wiring it into the real bucketing notebook. Swap `_blockkey` for the real bucket key once that notebook is ready.


In [59]:
# ---- Placeholder blocking (stand-in for the real embedding-bucket step in notebook 02) ----
def _blockkey(df):
    return df.with_columns(
        (pl.col("country") + "|" + pl.col("phonetic_code") + "|" + pl.col("core_name").str.slice(0, 4))
        .alias("bkey"))


def block_and_recall(s1_df, s2_df, s3_df, gt_df, label=""):
    s1n, s2n, s3n = normalize_frame(s1_df), normalize_frame(s2_df), normalize_frame(s3_df)
    s1b = _blockkey(s1n).select(["entity_id", "bkey"])
    cand = pl.concat([
        s1b.join(_blockkey(s2n).select(["entity_id", "bkey"]), on="bkey", how="inner", suffix="_c"),
        s1b.join(_blockkey(s3n).select(["entity_id", "bkey"]), on="bkey", how="inner", suffix="_c"),
    ]).select(["entity_id", "entity_id_c"]).rename({"entity_id": "s1", "entity_id_c": "cand"})
    cand_map = {r["s1"]: set(r["cands"]) for r in
                cand.group_by("s1").agg(pl.col("cand").unique().alias("cands")).to_dicts()}

    native_map = {r["entity_id"]: has_native_script(r["business_name"] or "")
                  for r in s1_df.select(["entity_id", "business_name"]).to_dicts()}
    truep = gt_df.filter(pl.col("matched_entity_ids").is_not_null() & (pl.col("matched_entity_ids") != "")) \
        .select([pl.col("source1_entity_id").alias("s1"),
                  pl.col("matched_entity_ids").str.split(",").alias("m")]).explode("m")

    hits = tot = nhits = ntot = lhits = ltot = 0
    for r in truep.to_dicts():
        s1id, m = r["s1"], r["m"]
        tot += 1
        native = native_map.get(s1id, False)
        ntot += int(native)
        ltot += int(not native)
        if m in cand_map.get(s1id, ()):
            hits += 1
            nhits += int(native)
            lhits += int(not native)
    out = {"label": label, "n_pairs": tot,
           "recall_overall": round(hits / tot, 4) if tot else None,
           "recall_native_script_s1": round(nhits / ntot, 4) if ntot else None,
           "n_native_script_s1": ntot,
           "recall_latin_s1": round(lhits / ltot, 4) if ltot else None,
           "n_latin_s1": ltot}
    print(f"[{label}] overall={out['recall_overall']} (n={tot}) | "
          f"native-script S1={out['recall_native_script_s1']} (n={ntot}) | "
          f"latin S1={out['recall_latin_s1']} (n={ltot})")
    return out


In [60]:
# ---- Recall at scale, hybrid ON vs OFF -- this is what actually answers "is it worth it" ----
def run_hybrid_scale_study(scales=(100_000,), distractor_ratio=3, seed=42):
    results = []
    global XLIT_MODEL
    saved_model = XLIT_MODEL
    for n in scales:
        s1_s, s2_s, s3_s, gt_s = build_anchored_sample(n_s1=n, distractor_ratio=distractor_ratio,
                                                        seed=seed, tag=f"n{n}")
        for hybrid_on in (False, True):
            XLIT_MODEL = saved_model if hybrid_on else None
            XLIT_CACHE.clear()
            label = f"n_s1={n} hybrid={'ON' if hybrid_on else 'OFF'}"
            results.append(block_and_recall(s1_s, s2_s, s3_s, gt_s, label=label))
    XLIT_MODEL = saved_model
    json.dump(results, open(OUT / "hybrid_recall_scale_results.json", "w"), indent=1)
    return results


# Start with just 100k while validating the engine. Add 1_000_000 once that looks right --
# it re-scans the full s2/s3 train files for distractors (still lazy/hashed, but bigger),
# and consider dropping distractor_ratio to 1 at that scale to keep the blocking join light.
SCALE_RESULTS = run_hybrid_scale_study(scales=(100_000,))


anchored sample n_s1=100000: s2=467051 (true=167051), s3=479104 (true=179104) -- every true match is guaranteed present
[n_s1=100000 hybrid=OFF] overall=0.7752 (n=346155) | native-script S1=None (n=0) | latin S1=0.7752 (n=346155)
[n_s1=100000 hybrid=ON] overall=0.8085 (n=346155) | native-script S1=None (n=0) | latin S1=0.8085 (n=346155)


**Note:** the per-country sampler just below (`process_sample_per_country`) is the original independent-per-file preview sampler -- fine for eyeballing output shape, **not** for recall claims (use `build_anchored_sample` above for that). Test files must be processed with `FULL_RUN=true` before final submission regardless of any dev-time sampling -- every test S1/S2/S3 entity must appear in the real run.


In [67]:
# ---- Run: per-country parquet (preprocessing.md §4 contract) ----
# NEW_COLS / normalize_frame now defined earlier (see "Frame-level normalization helper"
# cell above) so the anchored-sample recall cells can use them too.
def countries_of(src):
    return [r["country"] for r in pl.scan_csv(str(src), separator="\t",
            infer_schema_length=10000).select("country").unique().collect().to_dicts()]

def process_sample_per_country(src, stem, quota_each=40000):
    out = {}
    for c in countries_of(src):
        df = normalize_frame(pl.scan_csv(str(src), separator="\t", infer_schema_length=10000).filter(
            pl.col("country") == c).head(quota_each).collect())
        dst = OUT / f"{stem}_{c}.parquet"
        df.write_parquet(str(dst))
        out[c] = len(df)
        print(f"WROTE {dst.name} rows={len(df)} (sample)")
    return out

def process_full_per_country(src, stem, chunk=CHUNK):
    total = pl.scan_csv(str(src), separator="\t", infer_schema_length=10000).select(pl.len()).collect().item()
    tmpdir = OUT / f"_tmp_{stem}"
    tmpdir.mkdir(exist_ok=True)
    seen = set()
    for off in range(0, total, chunk):
        df = pl.scan_csv(str(src), separator="\t", infer_schema_length=10000).slice(off, chunk).collect()
        df = normalize_frame(df)
        for c in df["country"].unique().to_list():
            part = df.filter(pl.col("country") == c)
            p = tmpdir / f"{c}_{off}.parquet"
            part.write_parquet(str(p))
            seen.add(c)
        print(f"{Path(src).name} {off + len(df)}/{total}")
    out = {}
    for c in sorted(seen):
        full = pl.concat([pl.read_parquet(str(f))
                          for f in sorted(tmpdir.glob(f"{c}_*.parquet"))], how="diagonal")
        dst = OUT / f"{stem}_{c}.parquet"
        full.write_parquet(str(dst))
        out[c] = len(full)
        print(f"WROTE {dst.name} rows={len(full)}")
    for f in tmpdir.glob("*.parquet"):
        f.unlink()
    tmpdir.rmdir()
    return out

counts = {}
SPEC = [("s1tr", "train_s1"), ("s1te", "test_s1"), ("s2tr", "train_s2"), ("s3tr", "train_s3"),
        ("s2te", "test_s2"), ("s3te", "test_s3")]   # test S2/S3 are needed for the submission
for k, stem in SPEC:
    if FULL_RUN:
        counts[k] = process_full_per_country(P[k], stem)
    else:
        counts[k] = process_sample_per_country(P[k], stem)
json.dump(counts, open(OUT / "processed_counts.json", "w"), indent=1)
print("DONE:", counts)


WROTE train_s1_US.parquet rows=40000 (sample)
WROTE train_s1_India.parquet rows=40000 (sample)
WROTE test_s1_France.parquet rows=40000 (sample)
WROTE test_s1_US.parquet rows=40000 (sample)
WROTE test_s1_India.parquet rows=40000 (sample)
WROTE train_s2_US.parquet rows=40000 (sample)
WROTE train_s2_India.parquet rows=40000 (sample)
WROTE train_s3_US.parquet rows=40000 (sample)
WROTE train_s3_India.parquet rows=40000 (sample)
WROTE test_s2_France.parquet rows=40000 (sample)
WROTE test_s2_US.parquet rows=40000 (sample)
WROTE test_s2_India.parquet rows=40000 (sample)
WROTE test_s3_France.parquet rows=40000 (sample)
WROTE test_s3_US.parquet rows=40000 (sample)
WROTE test_s3_India.parquet rows=40000 (sample)
DONE: {'s1tr': {'US': 40000, 'India': 40000}, 's1te': {'France': 40000, 'US': 40000, 'India': 40000}, 's2tr': {'US': 40000, 'India': 40000}, 's3tr': {'US': 40000, 'India': 40000}, 's2te': {'France': 40000, 'US': 40000, 'India': 40000}, 's3te': {'France': 40000, 'US': 40000, 'India': 40000

In [69]:
# ---- Verify: before/after, FR inspection, schema freeze ----
stem = "train_s1"
pat = str(OUT / f"{stem}_US.parquet")
df = pl.read_parquet(pat).select(["business_name", "core_name", "legal_form", "name_lat",
    "business_address", "addr_norm", "house_no", "zip_pin_cp", "city", "state_code"]).head(5)
for r in df.to_dicts():
    print(safe(r["business_name"]), "->", safe(r["core_name"]), "| legal=", safe(r["legal_form"]),
          "| lat=", safe(r["name_lat"]))
    print("   ", safe(r["business_address"]), "->", safe(r["addr_norm"]),
          "| house=", r["house_no"], "pin=", r["zip_pin_cp"], "city=", safe(r["city"]), "st=", safe(r["state_code"]))
frf = str(OUT / "test_s1_France.parquet")
fr = pl.read_parquet(frf).select(["business_name", "core_name", "business_address", "addr_norm"]).head(3)
print("--- FR inspection (test S1 France) ---")
for r in fr.to_dicts():
    print(safe(r["business_name"]), "->", safe(r["core_name"]))
    print("   ", safe(r["business_address"]), "->", safe(r["addr_norm"]))
print("schema=", pl.read_parquet(pat).columns)
print("counts=", json.load(open(OUT / "processed_counts.json")))
print("verify OK")


"Orelee's Barbershop" -> "orelee's barbershop" | legal= '' | lat= "orelee's barbershop"
    '1795 Westchester Drive, High Point, NC' -> '1795 westchester drive high point nc' | house=  pin=  city= 'high point' st= 'NC'
'Prime Money' -> 'prime money' | legal= '' | lat= 'prime money'
    '17560 Ellis Road, Tahlequah, OK' -> '17560 ellis road tahlequah ok' | house=  pin= 17560 city= 'tahlequah' st= 'OK'
'B+ Retail Inc' -> 'b+ retail' | legal= 'inc' | lat= 'b+ retail'
    '1712 Montebello Avenue, Phoenix, AZ' -> '1712 montebello avenue phoenix az' | house=  pin=  city= 'phoenix' st= 'AZ'
'Christ Chapel' -> 'christ chapel' | legal= '' | lat= 'christ chapel'
    '2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD' -> '2100 cameron drive unit apartment g dundalk md' | house=  pin=  city= 'dundalk' st= 'MD'
'Custom Wealth Services LLC' -> 'custom wealth services' | legal= 'llc' | lat= 'custom wealth services'
    'OH, Columbus, 5559 Orville Avenue' -> 'oh columbus 5559 orville avenue' | house= 